In [1]:
from s4dd import S4forDenovoDesign

model_path = (
    "/home/kjana/kalyanashis/ML/S4-model/"
    "official_results/deliverables/"
    "designs-and-models/chemblv31/"
)

print("Loading pretrained S4 model...")

s4 = S4forDenovoDesign.from_file(model_path)

print("Pretrained S4 loaded successfully!")

Loading pretrained S4 model...


/home/kjana/kalyanashis/ML/S4-model/s4-for-de-novo-drug-design/s4dd/s4_for_denovo_design.py:244: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  s4_model.load_state_dict(torch

Pretrained S4 loaded successfully!


In [2]:
from rdkit import Chem

designs, lls = s4.design_molecules(
    n_designs=20,
    batch_size=10
)

valid = 0

for i, smiles in enumerate(designs, 1):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        valid += 1

    print(i, smiles)

print(f"\nValid molecules: {valid}/{len(designs)}")

TypeError: S4forDenovoDesign.design_molecules() missing 1 required positional argument: 'temperature'

In [3]:
import sys
import torch
import inspect
import traceback
import s4dd

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("S4 package:", s4dd.__file__)

print("\nSampling method signature:")
print(inspect.signature(s4.design_molecules))

try:
    designs, lls = s4.design_molecules(
        n_designs=2,
        batch_size=1,
        temperature=1.0
    )

    print("Generated molecules:", designs)

except Exception:
    traceback.print_exc()

Python: 3.10.19 (main, Oct 21 2025, 16:43:05) [GCC 11.2.0]
PyTorch: 2.5.1+cu121
CUDA available: True
S4 package: /home/kjana/kalyanashis/ML/S4-model/s4-for-de-novo-drug-design/s4dd/__init__.py

Sampling method signature:
(n_designs: int, batch_size: int, temperature: float) -> Tuple[List[str], List[float]]
Generated molecules: ['CC(=O)Nc1ccc(OCCNC(=O)Nc2cccc(Oc3ccccc3)c2)cc1', 'Cc1ncc([N+](=O)[O-])n1CC(=O)NN=Cc1ccc2c(c1)OCO2']


In [5]:
from rdkit import Chem
import pandas as pd

designs, lls = s4.design_molecules(
    n_designs=1000,
    batch_size=32,
    temperature=1.0
)

records = []

for i, (smiles, ll) in enumerate(zip(designs, lls), 1):
    mol = Chem.MolFromSmiles(smiles)

    records.append({
        "sample_id": i,
        "smiles": smiles,
        "canonical_smiles": (
            Chem.MolToSmiles(mol)
            if mol is not None else None
        ),
        "valid": mol is not None,
        "log_likelihood": float(ll)
    })

df = pd.DataFrame(records)

valid_df = df[df["valid"]]

print("Generated:", len(df))
print("Valid:", len(valid_df))
print("Validity (%):", round(100 * len(valid_df) / len(df), 2))
print("Unique valid:", valid_df["canonical_smiles"].nunique())

df.to_csv("S4_ChEMBL_pretrained_1000.csv", index=False)

[09:40:04] SMILES Parse Error: unclosed ring for input: 'NCCCCCNCCCCNc1cc2nc(-c3cccs3)nc3c2ccc1CN1CCc2ccccc21'
[09:40:04] non-ring atom 31 marked aromatic
[09:40:04] non-ring atom 31 marked aromatic
[09:40:04] SMILES Parse Error: unclosed ring for input: 'CC1C(=O)OC2COC(OC3CC4C(C)(C)CCCC4(C)C3CCC23C)C1CON1CCOCC1'
[09:40:04] Can't kekulize mol.  Unkekulized atoms: 9 10 11 12 28
[09:40:04] Can't kekulize mol.  Unkekulized atoms: 9 10 11 12 28
[09:40:04] SMILES Parse Error: ring closure 6 duplicates bond between atom 17 and atom 19 for input: 'C=C(c1ccccc1)C1C(O)C2(O)OC34C5CC6(O)C6=CC(O)CCC6(C)C4CCC5(C)C1(O2)C(OC(=O)c1ccccc1)C3O'
[09:40:04] Can't kekulize mol.  Unkekulized atoms: 4 5 6 7 8 9 10 11 12 13 14 15 16 18 20 21 22
[09:40:04] Can't kekulize mol.  Unkekulized atoms: 4 5 6 7 8 9 10 11 12 13 14 15 16 18 20 21 22
[09:40:04] SMILES Parse Error: unclosed ring for input: 'CCC1OC(=O)C(C)C(OC2OC(C)CC(N(C)C)C2O)C(C)(OC)CC(C)C(=O)C(C)C(OC2OC(C)CC(N(C)C)C2O)C(C)(O)CC(C)C(=O)'
[09:40:04] Can'

Generated: 1000
Valid: 968
Validity (%): 96.8
Unique valid: 968


[09:40:05] SMILES Parse Error: unclosed ring for input: 'CC12CC34C=CC(=O)OC3C(OC(=O)c3cccs3)C1CCC2=O'
[09:40:05] SMILES Parse Error: unclosed ring for input: 'CON=C(C(=O)NC1C(=O)N2C(C(=O)[O-])=C(C[n+]3[O-])CSC12)c1nsc(N)n1'
[09:40:05] SMILES Parse Error: extra open parentheses while parsing: CCN(CC)S(=O)(=O)c1ccc(C(=O)NCC(=O)O
[09:40:05] SMILES Parse Error: check for mistakes around position 22:
[09:40:05] CN(CC)S(=O)(=O)c1ccc(C(=O)NCC(=O)O
[09:40:05] ~~~~~~~~~~~~~~~~~~~~^
[09:40:05] SMILES Parse Error: Failed parsing SMILES 'CCN(CC)S(=O)(=O)c1ccc(C(=O)NCC(=O)O' for input: 'CCN(CC)S(=O)(=O)c1ccc(C(=O)NCC(=O)O'
[09:40:05] Can't kekulize mol.  Unkekulized atoms: 4 5 6
[09:40:05] Can't kekulize mol.  Unkekulized atoms: 4 5 6


In [6]:
from rdkit import Chem, RDLogger
import pandas as pd

RDLogger.DisableLog("rdApp.error")

designs, lls = s4.design_molecules(
    n_designs=1000,
    batch_size=32,
    temperature=1.0
)

records = []

for i, (smiles, ll) in enumerate(zip(designs, lls), 1):
    mol = Chem.MolFromSmiles(smiles)

    records.append({
        "sample_id": i,
        "smiles": smiles,
        "canonical_smiles": (
            Chem.MolToSmiles(mol, isomericSmiles=True)
            if mol is not None else None
        ),
        "valid": mol is not None,
        "log_likelihood": float(ll),
    })

df = pd.DataFrame(records)

valid_df = df[df["valid"]]

print("Generated:", len(df))
print("Valid:", len(valid_df))
print("Validity (%):", round(100 * len(valid_df) / len(df), 2))
print("Unique valid:", valid_df["canonical_smiles"].nunique())

df.to_csv("S4_ChEMBL_pretrained_1000.csv", index=False)

Generated: 1000
Valid: 965
Validity (%): 96.5
Unique valid: 965


In [7]:
from pathlib import Path
from s4dd import S4forDenovoDesign

model_dir = Path(
    "/home/kjana/kalyanashis/ML/S4-model/"
    "official_results/deliverables/"
    "designs-and-models/chemblv31"
)

assert (model_dir / "model.pt").is_file()
assert (model_dir / "init_arguments.json").is_file()

print("Checkpoint:", (model_dir / "model.pt").resolve())
print("Checkpoint size:", (model_dir / "model.pt").stat().st_size)

s4 = S4forDenovoDesign.from_file(str(model_dir))

print("Official ChEMBL S4 checkpoint loaded successfully.")

Checkpoint: /home/kjana/kalyanashis/ML/S4-model/official_results/deliverables/designs-and-models/chemblv31/model.pt
Checkpoint size: 13285577


/home/kjana/kalyanashis/ML/S4-model/s4-for-de-novo-drug-design/s4dd/s4_for_denovo_design.py:244: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  s4_model.load_state_dict(torch

Official ChEMBL S4 checkpoint loaded successfully.


In [1]:
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs

RDLogger.DisableLog("rdApp.error")

df = pd.read_csv(
    "experiments/s4_ppi786/s4_generated_1000.csv"
)

print("Columns:", df.columns.tolist())

smiles_col = (
    "canonical_smiles"
    if "canonical_smiles" in df.columns
    else "smiles"
)

# Omeprazole-related sulfoxide-benzimidazole motif.
# Accept neutral and charge-separated sulfoxide representations.
queries = [
    Chem.MolFromSmarts(
        "[#6]-[S;X3](=[O])-[c]1nc2ccccc2[nH]1"
    ),
    Chem.MolFromSmarts(
        "[#6]-[S+;X3]([O-])-[c]1nc2ccccc2[nH]1"
    ),
]

def check_core(smiles):
    if pd.isna(smiles):
        return False

    mol = Chem.MolFromSmiles(str(smiles))

    if mol is None:
        return False

    return any(
        mol.HasSubstructMatch(q)
        for q in queries
        if q is not None
    )

df["ppi_core"] = df[smiles_col].apply(check_core)

valid = df[df["valid"].astype(bool)].copy()
unique_valid = valid.drop_duplicates(smiles_col)

core = unique_valid[unique_valid["ppi_core"]]

print("Generated:", len(df))
print("Valid:", len(valid))
print("Unique valid:", len(unique_valid))
print("Unique core-retaining:", len(core))
print(
    "Core retention among unique valid (%):",
    round(100 * len(core) / max(len(unique_valid), 1), 2)
)

core.to_csv(
    "experiments/s4_ppi786/"
    "s4_unique_ppi_core.csv",
    index=False
)

Columns: ['sample_id', 'raw_smiles', 'canonical_smiles', 'valid', 'unique_valid', 'mean_log_likelihood']
Generated: 1000
Valid: 959
Unique valid: 952
Unique core-retaining: 37
Core retention among unique valid (%): 3.89


In [2]:
from pathlib import Path
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, Crippen, QED
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

RDLogger.DisableLog("rdApp.error")

s4_root = Path.cwd()
gnn_root = Path(
    "/home/kjana/kalyanashis/ML/alfatrainig/"
    "deep_learning_bootcamp/project/GNN"
)

files = {
    "Transformer-786":
        gnn_root / "generation/transformer_ppi_finetuned_evaluated.csv",
    "S4-771":
        s4_root / "experiments/s4_ppi786/s4_generated_1000.csv",
    "S4-247":
        s4_root / "experiments/s4_ppi248/s4_generated_1000.csv"
}

# Broad core: sulfoxide bonded to benzimidazole
broad_patterns = [
    Chem.MolFromSmarts(
        "[#6]-[S](=[O])-[c]1nc2ccccc2[nH]1"
    ),
    Chem.MolFromSmarts(
        "[#6]-[S+]([O-])-[c]1nc2ccccc2[nH]1"
    )
]

# Strict core: 2-pyridylmethyl-sulfoxide-benzimidazole
strict_patterns = [
    Chem.MolFromSmarts(
        "[S](=O)([CH2]c1ncccc1)c1nc2ccccc2[nH]1"
    ),
    Chem.MolFromSmarts(
        "[S+]([O-])([CH2]c1ncccc1)c1nc2ccccc2[nH]1"
    )
]

def match_any(mol, patterns):
    return any(mol.HasSubstructMatch(p) for p in patterns)

def evaluate_model(path, model_name):
    df = pd.read_csv(path)

    smiles_col = next(
        c for c in [
            "canonical_smiles_checked",
            "canonical_smiles",
            "smiles",
            "raw_smiles"
        ]
        if c in df.columns
    )

    records = []

    for idx, smi in enumerate(df[smiles_col], 1):
        mol = Chem.MolFromSmiles(str(smi)) if pd.notna(smi) else None

        if mol is None:
            records.append({
                "model": model_name,
                "sample_id": idx,
                "valid": False
            })
            continue

        records.append({
            "model": model_name,
            "sample_id": idx,
            "valid": True,
            "canonical_smiles": Chem.MolToSmiles(mol),
            "broad_core": match_any(mol, broad_patterns),
            "strict_core": match_any(mol, strict_patterns),
            "MW": Descriptors.MolWt(mol),
            "LogP": Crippen.MolLogP(mol),
            "TPSA": rdMolDescriptors.CalcTPSA(mol),
            "QED": QED.qed(mol),
            "scaffold": MurckoScaffold.MurckoScaffoldSmiles(
                mol=mol
            )
        })

    result = pd.DataFrame(records)
    valid = result[result["valid"]]
    unique = valid.drop_duplicates("canonical_smiles")

    summary = {
        "model": model_name,
        "generated": len(result),
        "valid": len(valid),
        "validity_pct": round(100 * len(valid) / len(result), 2),
        "unique_valid": len(unique),
        "broad_core_unique": int(unique["broad_core"].sum()),
        "strict_core_unique": int(unique["strict_core"].sum()),
        "unique_scaffolds": unique["scaffold"].nunique()
    }

    return result, summary

all_results = []
summaries = []

for name, path in files.items():
    result, summary = evaluate_model(path, name)
    all_results.append(result)
    summaries.append(summary)

comparison = pd.DataFrame(summaries)
evaluated = pd.concat(all_results, ignore_index=True)

display(comparison)

out = s4_root / "experiments/comparison"
out.mkdir(parents=True, exist_ok=True)

comparison.to_csv(out / "model_summary.csv", index=False)
evaluated.to_csv(out / "all_evaluated.csv", index=False)

,model,generated,valid,validity_pct,unique_valid,broad_core_unique,strict_core_unique,unique_scaffolds
0,Transformer-786,1000,988,98.8,820,124,79,519
1,S4-771,1000,959,95.9,952,40,0,873
2,S4-247,1000,957,95.7,957,0,0,903


In [3]:
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator

ref = pd.read_csv(gnn_root / "PPI_molecule_summary_248.csv")

fpgen = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=2048
)

reference_fps = []

for smi in ref["canonical_smiles"].dropna():
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        reference_fps.append(fpgen.GetFingerprint(mol))

def maximum_ppi_similarity(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fp = fpgen.GetFingerprint(mol)
    return max(
        DataStructs.BulkTanimotoSimilarity(fp, reference_fps)
    )

valid_results = evaluated[evaluated["valid"]].copy()

valid_results["max_PPI_similarity"] = (
    valid_results["canonical_smiles"]
    .apply(maximum_ppi_similarity)
)

display(
    valid_results.groupby("model")["max_PPI_similarity"]
    .describe().round(3)
)

valid_results.to_csv(
    out / "molecular_similarity.csv",
    index=False
)

,count,mean,std,min,25%,50%,75%,max
model,,,,,,,,
S4-247,957.0,0.218,0.057,0.067,0.183,0.211,0.243,0.684
S4-771,959.0,0.248,0.072,0.062,0.198,0.235,0.287,0.553
Transformer-786,988.0,0.416,0.192,0.093,0.296,0.367,0.482,1.000


In [5]:
import torch

checkpoint = (
    "/home/kjana/kalyanashis/ML/S4-model/"
    "official_results/deliverables/"
    "designs-and-models/chemblv31/model.pt"
)

state = torch.load(
    checkpoint,
    map_location="cpu",
    weights_only=True
)

print("Total checkpoint parameters:")
print(sum(t.numel() for t in state.values()))

print("\nFirst 30 parameter names:")
for name, tensor in list(state.items())[:30]:
    print(name, tuple(tensor.shape))

Total checkpoint parameters:
3310663

First 30 parameter names:
embedding.weight (37, 256)
model.layers.0.layer.D (1, 256)
model.layers.0.layer.kernel.kernel.C (1, 256, 32, 2)
model.layers.0.layer.kernel.kernel.log_dt (256,)
model.layers.0.layer.kernel.kernel.B (1, 1, 32, 2)
model.layers.0.layer.kernel.kernel.P (1, 1, 32, 2)
model.layers.0.layer.kernel.kernel.inv_w_real (1, 32)
model.layers.0.layer.kernel.kernel.w_imag (1, 32)
model.layers.0.layer.kernel.kernel.L ()
model.layers.0.layer.output_linear.0.weight (512, 256)
model.layers.0.layer.output_linear.0.bias (512,)
model.layers.0.norm.norm.m (1,)
model.layers.0.norm.norm.s (1,)
model.layers.1.layer.D (1, 256)
model.layers.1.layer.kernel.kernel.C (1, 256, 32, 2)
model.layers.1.layer.kernel.kernel.log_dt (256,)
model.layers.1.layer.kernel.kernel.B (1, 1, 32, 2)
model.layers.1.layer.kernel.kernel.P (1, 1, 32, 2)
model.layers.1.layer.kernel.kernel.inv_w_real (1, 32)
model.layers.1.layer.kernel.kernel.w_imag (1, 32)
model.layers.1.layer.